# Testing

In [1]:
test_prompt = """
Analyze this image. Does it contain hateful content?

Categories:
- homophobic: anti-gay/lesbian hate
- transphobic: anti-transgender hate
- misogynistic: anti-women hate
- none: neutral, unrelated, or no hateful content present

Describe the image and provide which category it belongs to
"""

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key="",
)

# completion = client.chat.completions.create(
#     model="Qwen/Qwen3-VL-8B-Instruct:fastest",
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "text",
#                     "text": test_prompt
#                 },
#                 {
#                     "type": "image_url",
#                     "image_url": {
#                         "url": "https://drive.google.com/uc?export=download&id=1W9OvSWj4hqSaMyzihQnADw59Zu4Vxw-c"
#                     }
#                 }
#             ]
#         }
#     ],
# )

# print(completion.choices[0].message)

/home/v01dfraktal/tan/NUIG/PROJ/dataset/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Tamil

In [3]:
prompt = """Analyze this image. Does it contain hateful content?

Categories:
- homophobic: anti-gay/lesbian hate
- transphobic: anti-transgender hate
- misogynistic: anti-women hate
- NA: neutral, unrelated, or no hateful content present

Answer only with the category name.
"""

def classify_image(file_id):

    completion = client.chat.completions.create(
        model="Qwen/Qwen3-VL-8B-Instruct:fastest",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"https://drive.google.com/uc?export=download&id={file_id}"
                        }
                    }
                ]
            }
        ],
    )

    return completion.choices[0].message.content


In [4]:
import json

with open("misogyny/tamil/misogyny_tamil_drive.json", "r") as f:
    files = json.load(f)

In [5]:
import time

rows = []
BATCH_SIZE = 40
WAIT_SECONDS = 60

for batch_start in range(0, len(files), BATCH_SIZE):
    batch = files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        try:
            label = classify_image(file_id)
        except Exception as e:
            label = f"ERROR: {e}"
        print(label)
        rows.append({"image_id": f["name"], "labels": label})
        time.sleep(2)

    # Wait between batches, but not after the last one
    if batch_start + BATCH_SIZE < len(files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)


--- Batch 1 (1 to 40) ---
[1/356] Processing 1262.jpg ... NA
[2/356] Processing 744.jpg ... NA
[3/356] Processing 1010.jpg ... NA
[4/356] Processing 1237.jpg ... NA
[5/356] Processing 620.jpg ... NA
[6/356] Processing 1627.jpg ... NA
[7/356] Processing 176.jpg ... NA
[8/356] Processing 555.jpg ... NA
[9/356] Processing 1719.jpg ... NA
[10/356] Processing 1109.jpg ... NA
[11/356] Processing 530.jpg ... NA
[12/356] Processing 1418.jpg ... NA
[13/356] Processing 1028.jpg ... misogynistic
[14/356] Processing 1738.jpg ... NA
[15/356] Processing 507.jpg ... NA
[16/356] Processing 1054.jpg ... ERROR: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69ad9fca-68ca2b6f122369c744af9208;a91ab565-4ca5-4e41-a256-531f360372b8)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Altern

KeyboardInterrupt: 

In [6]:
import csv
OUTPUT_CSV = "misogyny/tamil/test_pred_qwen3_tamil.csv"
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Saved {len(rows)} rows → {OUTPUT_CSV}")


Done! Saved 315 rows → misogyny/tamil/test_pred_qwen3_tamil.csv


In [ ]:
# Build a set of image_ids that need reprocessing
error_ids = {r["image_id"] for r in rows if str(r["labels"]).startswith("ERROR:")}

# Filter files to only those with errors
retry_files = [f for f in files if f["name"] in error_ids]

In [10]:
# Build a set of image_ids that need reprocessing
error_ids = {r["image_id"] for r in rows if str(r["labels"]).startswith("ERROR:")}

# Filter files to only those with errors
retry_files = [f for f in files if f["name"] in error_ids]

new_rows = []

print(f"Retrying {len(retry_files)} failed images...")

BATCH_SIZE = 40
WAIT_SECONDS = 60

for batch_start in range(0, len(retry_files), BATCH_SIZE):
    batch = retry_files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(retry_files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        try:
            label = classify_image(file_id)
        except Exception as e:
            label = f"ERROR: {e}"
        print(label)

        # Update existing row instead of appending a new one
        for r in rows:
            if r["image_id"] == f["name"]:
                r["labels"] = label
                break

        time.sleep(2)

    if batch_start + BATCH_SIZE < len(retry_files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)

print(f"\nDone. {sum(1 for r in rows if str(r['labels']).startswith('ERROR:'))} errors remaining.")

Retrying 143 failed images...

--- Batch 1 (1 to 40) ---
[1/143] Processing 1054.jpg ... NA
[2/143] Processing 1236.jpg ... NA
[3/143] Processing 395.jpg ... NA
[4/143] Processing 1666.jpg ... NA
[5/143] Processing 301.jpg ... NA
[6/143] Processing 428.jpg ... NA
[7/143] Processing 1162.jpg ... ERROR: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/v1/chat/completions' (Amz CF ID: snSo-YY58J4Ixwc-0zfxna34IIsHXit4YSYZY71Qqu46awISlc1BXg==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504
[8/143] Processing 701.jpg ... NA
[9/143] Processing 251.jpg ... NA
[10/143] Processing 1290.jpg ... NA
[11/143] Processing 1349.jpg ... NA
[12/143] Processing 1660.jpg ... NA
[13/143] Processing 515.jpg ... NA
[14/143] Processing 1205.jpg ... NA
[15/143] Processing 1659.jpg ... NA
[16/143] Processing 911.jpg ... NA
[17/143] Processing 1226.jpg ... NA
[18/143] Processing 52.jpg ... NA
[19/143] Processing 416.jpg ... NA
[20/143] Processing

In [11]:
# Build a set of image_ids that need reprocessing
error_ids = {r["image_id"] for r in rows if str(r["labels"]).startswith("ERROR:")}

# Filter files to only those with errors
retry_files = [f for f in files if f["name"] in error_ids]

new_rows = []

print(f"Retrying {len(retry_files)} failed images...")

BATCH_SIZE = 40
WAIT_SECONDS = 60

for batch_start in range(0, len(retry_files), BATCH_SIZE):
    batch = retry_files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(retry_files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        try:
            label = classify_image(file_id)
        except Exception as e:
            label = f"ERROR: {e}"
        print(label)

        # Update existing row instead of appending a new one
        for r in rows:
            if r["image_id"] == f["name"]:
                r["labels"] = label
                break

        time.sleep(2)

    if batch_start + BATCH_SIZE < len(retry_files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)

print(f"\nDone. {sum(1 for r in rows if str(r['labels']).startswith('ERROR:'))} errors remaining.")

Retrying 3 failed images...

--- Batch 1 (1 to 3) ---
[1/3] Processing 1162.jpg ... NA
[2/3] Processing 60.jpg ... NA
[3/3] Processing 70.jpg ... NA

Done. 0 errors remaining.


In [15]:
import csv
OUTPUT_CSV = "misogyny/tamil/test_pred_qwen3_tamil_2.csv"
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Saved {len(rows)} rows → {OUTPUT_CSV}")


Done! Saved 315 rows → misogyny/tamil/test_pred_qwen3_tamil_2.csv


In [20]:
existing_image_ids = {item['image_id'] for item in rows}

retry_files = [item for item in files if item['name'] not in existing_image_ids]

print(f"Retrying {len(retry_files)} failed images...")

BATCH_SIZE = 40
WAIT_SECONDS = 60

for batch_start in range(0, len(retry_files), BATCH_SIZE):
    batch = retry_files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(retry_files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        try:
            label = classify_image(file_id)
        except Exception as e:
            label = f"ERROR: {e}"
        print(label)

        rows.append({"image_id": f["name"], "labels": label})

        time.sleep(2)

    if batch_start + BATCH_SIZE < len(retry_files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)

print(f"\nDone. {sum(1 for r in rows if str(r['labels']).startswith('ERROR:'))} errors remaining.")

Retrying 41 failed images...

--- Batch 1 (1 to 40) ---
[1/41] Processing 382.jpg ... NA
[2/41] Processing 1366.jpg ... NA
[3/41] Processing 830.jpg ... NA
[4/41] Processing 549.jpg ... NA
[5/41] Processing 943.jpg ... NA
[6/41] Processing 1711.jpg ... NA
[7/41] Processing 325.jpg ... NA
[8/41] Processing 600.jpg ... NA
[9/41] Processing 1269.jpg ... NA
[10/41] Processing 742.jpg ... ERROR: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/v1/chat/completions' (Amz CF ID: IGNeIneYG91ZgJ4K7QPujhTqrJhFu6lVjdq0Xnr-O1A1OEMhIH56hg==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504
[11/41] Processing 1081.jpg ... misogynistic
[12/41] Processing 1272.jpg ... NA
[13/41] Processing 221.jpg ... NA
[14/41] Processing 304.jpg ... NA
[15/41] Processing 1141.jpg ... NA
[16/41] Processing 562.jpg ... NA
[17/41] Processing 882.jpg ... NA
[18/41] Processing 786.jpg ... NA
[19/41] Processing 949.jpg ... NA
[20/41] Processing 591.jpg ... N

In [21]:
len(rows)

356

In [22]:
# Build a set of image_ids that need reprocessing
error_ids = {r["image_id"] for r in rows if str(r["labels"]).startswith("ERROR:")}

# Filter files to only those with errors
retry_files = [f for f in files if f["name"] in error_ids]

new_rows = []

print(f"Retrying {len(retry_files)} failed images...")

BATCH_SIZE = 40
WAIT_SECONDS = 60

for batch_start in range(0, len(retry_files), BATCH_SIZE):
    batch = retry_files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(retry_files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        try:
            label = classify_image(file_id)
        except Exception as e:
            label = f"ERROR: {e}"
        print(label)

        # Update existing row instead of appending a new one
        for r in rows:
            if r["image_id"] == f["name"]:
                r["labels"] = label
                break

        time.sleep(2)

    if batch_start + BATCH_SIZE < len(retry_files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)

print(f"\nDone. {sum(1 for r in rows if str(r['labels']).startswith('ERROR:'))} errors remaining.")

Retrying 2 failed images...

--- Batch 1 (1 to 2) ---
[1/2] Processing 742.jpg ... NA
[2/2] Processing 1388.jpg ... NA

Done. 0 errors remaining.


In [23]:
import csv
OUTPUT_CSV = "misogyny/tamil/test_pred_qwen3_tamil.csv"
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Saved {len(rows)} rows → {OUTPUT_CSV}")


Done! Saved 356 rows → misogyny/tamil/test_pred_qwen3_tamil.csv


# Malayalam

In [24]:
with open("misogyny/malayalam/misogyny_malayalam_drive.json", "r") as f:
    files = json.load(f)

In [25]:
len(files)

200

In [26]:
rows = []
BATCH_SIZE = 40
WAIT_SECONDS = 60
MAX_RETRIES = 2

for batch_start in range(0, len(files), BATCH_SIZE):
    batch = files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        for attempt in range(1, MAX_RETRIES + 2):  # 1 original + 2 retries
            try:
                label = classify_image(file_id)
            except Exception as e:
                label = f"ERROR: {e}"

            if not label.startswith("ERROR:"):
                break

            if attempt <= MAX_RETRIES:
                print(f"\n  Retry {attempt}/{MAX_RETRIES} for {f['name']}...", end=" ", flush=True)
                time.sleep(2)

        print(label)
        rows.append({"image_id": f["name"], "labels": label})
        time.sleep(2)

    # Wait between batches, but not after the last one
    if batch_start + BATCH_SIZE < len(files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)


--- Batch 1 (1 to 40) ---
[1/200] Processing 79.jpg ... NA
[2/200] Processing 614.jpg ... NA
[3/200] Processing 209.jpg ... NA
[4/200] Processing 333.jpg ... NA
[5/200] Processing 409.jpg ... NA
[6/200] Processing 214.jpg ... NA
[7/200] Processing 200.jpg ... NA
[8/200] Processing 657.jpg ... NA
[9/200] Processing 830.jpg ... NA
[10/200] Processing 949.jpg ... NA
[11/200] Processing 750.jpg ... NA
[12/200] Processing 943.jpg ... NA
[13/200] Processing 366.jpg ... NA
[14/200] Processing 356.jpg ... NA
[15/200] Processing 138.jpg ... NA
[16/200] Processing 846.jpg ... NA
[17/200] Processing 527.jpg ... NA
[18/200] Processing 378.jpg ... NA
[19/200] Processing 671.jpg ... NA
[20/200] Processing 495.jpg ... NA
[21/200] Processing 678.jpg ... NA
[22/200] Processing 803.jpg ... NA
[23/200] Processing 260.jpg ... NA
[24/200] Processing 255.jpg ... NA
[25/200] Processing 364.jpg ... NA
[26/200] Processing 487.jpg ... NA
[27/200] Processing 990.jpg ... NA
[28/200] Processing 111.jpg ... NA
[29

In [27]:
correct_rows = [row for row in rows if not row["labels"].startswith("ERROR:")]

len(correct_rows)

200

In [28]:
OUTPUT_CSV = "misogyny/malayalam/test_pred_qwen3_malayalam.csv"
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(correct_rows)

print(f"\nDone! Saved {len(correct_rows)} rows → {OUTPUT_CSV}")


Done! Saved 200 rows → misogyny/malayalam/test_pred_qwen3_malayalam.csv
